# 5차시 (2) RAG 챗봇 만들기 — 실습

한경국립대 2026 여름특강 · 딥러닝/머신러닝 입문 (초급반)

## RAG가 뭔가요?

앞 노트북 1-2에서, 답의 근거가 될 텍스트를 **직접 프롬프트에 붙여넣어** 주면 모델이 더 정확히 답하는 걸 봤습니다.
하지만 문서가 100쪽이라면? 매번 통째로 붙여넣을 수는 없습니다.

**RAG (Retrieval-Augmented Generation, 검색 증강 생성)** = 질문이 들어오면
1. 내 문서 더미에서 **질문과 관련된 부분만 자동으로 찾아(Retrieval)**,
2. 그 부분을 근거로 **LLM이 답을 생성(Generation)** 하는 방식입니다.

> 한 줄 요약: **"내 문서에서 관련 내용을 찾아서, 그걸 보고 답하는 챗봇".**
> 우리가 잘 아는 **Google NotebookLM** 도 이런 소스 기반 도구입니다(맨 끝에서 다시 언급).

> 📌 **✏️ 표시가 붙은 "연습문제"는 직접 풀어 보는 칸입니다.** 먼저 스스로 작성해 본 뒤 정답을 확인하세요.

## 오늘 만들 RAG 파이프라인 (6단계)

```
[1] 문서 적재   원본 문서를 불러온다
      ↓
[2] 청킹       긴 문서를 작은 조각(chunk)으로 자른다
      ↓
[3] 임베딩     각 조각을 '의미 좌표' 숫자 벡터로 바꾼다
      ↓
[4] 벡터 저장   벡터들을 벡터DB(Chroma)에 넣는다
      ↓
[5] 검색       질문도 벡터로 바꿔, 가장 비슷한 조각 top-k를 찾는다 (코사인 유사도)
      ↓
[6] 생성       찾은 조각을 근거로 LLM이 최종 답을 만든다
```

아래에서 **한 단계씩 셀을 나눠** 직접 만들어 보고, 마지막에 **Gradio 챗봇**으로 합칩니다.

> 3차시에서 배운 **벡터·코사인 유사도** 가 [3]·[5]단계에서 그대로 쓰입니다. "비슷한 의미 = 가까운 벡터".

---
## 0. 환경 세팅

### 0-1. 라이브러리 설치
RAG에 필요한 도구를 설치합니다(1~2분).
- `langchain` 계열: 문서 적재·청킹·검색·LLM 연결을 묶어 주는 프레임워크
- `langchain-chroma` / `chromadb`: 벡터를 저장·검색하는 가벼운 **벡터DB**
- `langchain-openai`: OpenAI 임베딩·LLM 호출

설치 후 빨간 의존성 경고가 떠도 무시해도 됩니다.

In [ ]:
!pip install -qU langchain langchain-openai langchain-community
!pip install -qU langchain-chroma chromadb
!pip install -qU "gradio>=5.0"
print("설치 완료!")

### 0-2. OpenAI API 키 입력
주최측에서 받은 키를 붙여넣고 Enter (화면에 보이지 않습니다).

In [ ]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API 키를 입력하세요: ")

print("API 키 설정 완료" if os.environ.get("OPENAI_API_KEY") else "키가 비어 있습니다.")

---
## [1] 문서 적재 — 답변의 근거가 될 문서 준비

실습이니 외부 파일 없이, **가상의 사내 안내 문서**를 코드로 직접 만들어 `docs/` 폴더에 `.txt` 로 저장합니다.
(실제로는 PDF·웹페이지·회사 위키 등을 불러옵니다. 형식만 다를 뿐 이후 단계는 동일합니다.)

주제: *한경 테크 신입사원 안내서* — 휴가, 재택, 장비, 식대 같은 사내 규정.

In [ ]:
import os

os.makedirs("docs", exist_ok=True)

documents = {
    "휴가.txt": """[연차 휴가 안내]
신입사원은 입사 첫 해에 11일의 연차가 부여됩니다.
연차는 입사 1년 후부터 매년 15일로 늘어나며, 3년차부터 2년마다 1일씩 추가됩니다.
연차 신청은 최소 3일 전에 그룹웨어에서 신청해야 합니다.
반차(반일 휴가)도 사용할 수 있으며, 오전 반차와 오후 반차로 나뉩니다.""",

    "재택근무.txt": """[재택근무 규정]
한경 테크는 주 2회까지 재택근무가 가능합니다.
재택근무를 하려면 전날 오후 6시까지 팀장에게 신청해야 합니다.
재택근무 시에도 코어타임(오전 10시 ~ 오후 4시)에는 메신저에 접속해 있어야 합니다.
신입사원은 입사 후 3개월(수습 기간)이 지나야 재택근무를 신청할 수 있습니다.""",

    "장비.txt": """[업무 장비 지원]
전 직원에게 노트북 1대가 지급됩니다. 개발 직군은 메모리 32GB 모델을 받습니다.
외부 모니터는 1인당 최대 2대까지 지원됩니다.
키보드와 마우스는 자유롭게 신청할 수 있으며, 신청 후 3일 이내에 지급됩니다.
장비가 고장 나면 IT 헬프데스크(내선 1234)로 문의하면 됩니다.""",

    "식대.txt": """[식대 및 복지]
점심 식대로 매일 1만 원이 지원되며, 사내 식당 또는 제휴 식당에서 사용할 수 있습니다.
야근 시(오후 8시 이후) 저녁 식대 1만 원이 추가로 지원됩니다.
매월 마지막 주 금요일은 '패밀리 데이'로 오후 5시에 조기 퇴근합니다.
사내 카페에서는 모든 음료를 50% 할인된 가격에 구매할 수 있습니다.""",
}

for filename, content in documents.items():
    with open(os.path.join("docs", filename), "w", encoding="utf-8") as f:
        f.write(content)

print("문서 저장 완료:", os.listdir("docs"))

이제 저장한 `.txt` 들을 LangChain의 **로더(loader)** 로 불러옵니다.
`DirectoryLoader` 는 폴더 안 파일들을 한 번에 읽어 `Document` 객체 목록으로 만들어 줍니다.
각 `Document` 는 본문(`page_content`)과 출처 정보(`metadata`)를 가집니다.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "docs",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
raw_docs = loader.load()

print(f"불러온 문서 수: {len(raw_docs)}개\n")
print("첫 문서 출처:", raw_docs[0].metadata)
print("첫 문서 내용 미리보기:\n", raw_docs[0].page_content[:80], "...")

---
## [2] 청킹 — 긴 문서를 작은 조각으로 자르기

**왜 자르나요?**
- 검색은 "질문과 관련된 *부분*"을 찾는 것이라, 문서가 통째로 한 덩어리면 관련 없는 내용까지 딸려옵니다.
- 작게 자르면(=chunk) 더 **정확한 부분**만 골라 LLM에 줄 수 있습니다.

`RecursiveCharacterTextSplitter` 를 씁니다.
- `chunk_size` : 한 조각의 최대 글자 수
- `chunk_overlap` : 조각끼리 겹치는 글자 수 (문장이 경계에서 잘려 맥락이 끊기는 걸 줄임)

우리 문서는 짧으니 작은 값으로 설정합니다(실무 문서는 보통 500~1000자).

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,     # 한 조각 최대 150자
    chunk_overlap=20,   # 앞뒤 20자씩 겹치기
)
chunks = splitter.split_documents(raw_docs)

print(f"원본 {len(raw_docs)}개 문서 -> {len(chunks)}개 조각으로 분할\n")
for i, c in enumerate(chunks[:3]):
    print(f"--- 조각 {i} (출처: {os.path.basename(c.metadata['source'])}) ---")
    print(c.page_content)
    print()

---
## [3] 임베딩 — 글자를 '의미 좌표(벡터)'로 바꾸기

컴퓨터는 글자의 *의미* 를 직접 비교하지 못합니다. 그래서 각 조각을 **숫자 벡터**로 바꿉니다(3차시에서 본 그 벡터!).
핵심 성질: **의미가 비슷한 문장 → 벡터도 가까이** 위치합니다.

OpenAI의 `text-embedding-3-small` 모델을 씁니다(저렴하고 빠름).
먼저 임베딩이 실제로 어떤 숫자인지, 그리고 "비슷한 문장은 가깝다"는 성질을 **직접 확인**해 봅시다.

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 문장 하나를 벡터로 바꿔 보기
vec = embeddings.embed_query("연차 휴가는 며칠인가요?")
print("벡터 길이(차원 수):", len(vec))
print("앞부분 미리보기:", [round(x, 3) for x in vec[:8]], "...")

### 잠깐 실험: '비슷한 의미 = 가까운 벡터' 확인하기
두 문장의 벡터가 얼마나 가까운지 **코사인 유사도**로 잽니다(3차시 내용).
- 값이 **1에 가까울수록 비슷한 의미**, 0에 가까울수록 무관.

'휴가' 관련 질문은 '연차' 문장과 가깝고, '점심' 문장과는 멀어야 정상입니다.

In [ ]:
import numpy as np

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

query_vec = embeddings.embed_query("휴가는 며칠 쓸 수 있나요?")
sent_a_vec = embeddings.embed_query("신입사원은 첫 해에 연차 11일을 받습니다.")
sent_b_vec = embeddings.embed_query("점심 식대로 매일 1만 원이 지원됩니다.")

print("질문 vs '연차 11일' 문장  유사도:", round(cosine_similarity(query_vec, sent_a_vec), 3))
print("질문 vs '점심 식대' 문장  유사도:", round(cosine_similarity(query_vec, sent_b_vec), 3))
print("\n=> 휴가 질문은 연차 문장과 더 가깝습니다. 이 성질로 '관련 조각'을 찾습니다.")

---
## [4] 벡터 저장 — 벡터DB(Chroma)에 모든 조각 넣기

조각이 수천 개면 매번 직접 유사도를 계산하긴 번거롭습니다. **벡터DB** 가 이 일을 대신해 줍니다.
- 조각들의 벡터를 저장해 두고,
- 질문 벡터가 들어오면 **가장 가까운 조각들을 빠르게 찾아** 줍니다.

`Chroma.from_documents` 한 줄이면 [3]임베딩 + [4]저장을 함께 처리합니다(각 조각을 임베딩해서 DB에 넣음).

In [ ]:
from langchain_chroma import Chroma

# chunks(조각들)를 임베딩해서 Chroma 벡터DB에 저장
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="hankyung_handbook",
)

print("벡터DB 생성 완료! 저장된 조각 수:", vectorstore._collection.count())

---
## [5] 검색 — 질문과 가장 비슷한 조각 top-k 찾기

이제 질문을 던지면 벡터DB가 **가장 관련 있는 조각 k개**를 코사인 유사도로 골라 줍니다(이를 **top-k 검색** 이라 합니다).
`similarity_search(질문, k=...)` 로 직접 확인해 봅시다.

In [ ]:
query = "재택근무는 일주일에 몇 번 할 수 있어?"

found = vectorstore.similarity_search(query, k=2)   # 가장 비슷한 조각 2개

print(f"질문: {query}\n")
print("=== 검색된 관련 조각 ===")
for i, doc in enumerate(found):
    print(f"[{i+1}] (출처: {os.path.basename(doc.metadata['source'])})")
    print(doc.page_content, "\n")

**TODO 1.** `query` 를 다른 질문으로 바꿔, 관련 있는 조각이 잘 검색되는지 확인해 보세요.
(예: "노트북 메모리는 얼마야?", "패밀리 데이가 뭐야?")

In [ ]:
# TODO: 아래 질문을 바꿔 검색 결과를 확인해 보세요.
my_query = "노트북은 어떤 걸 받아?"
for i, doc in enumerate(vectorstore.similarity_search(my_query, k=2)):
    print(f"[{i+1}]", doc.page_content, "\n")

---
## [6] 생성 — 찾은 조각을 근거로 LLM이 답하기

드디어 마지막 단계입니다. [5]에서 찾은 조각들을 **프롬프트에 근거로 넣고** LLM에게 답을 만들게 합니다.
노트북 1의 "참고 텍스트 제공" 과 똑같은 원리인데, 그 텍스트를 **자동 검색으로 채운다**는 점만 다릅니다.

핵심: 프롬프트에서 *"아래 참고 자료만 근거로 답하고, 자료에 없으면 모른다고 답해"* 라고 못 박아 **환각을 줄입니다**.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def rag_answer(question, k=3):
    # 1) 관련 조각 검색
    docs = vectorstore.similarity_search(question, k=k)
    context = "\n\n".join(d.page_content for d in docs)

    # 2) 검색 결과를 근거로 넣은 프롬프트 구성
    prompt = f"""너는 회사 안내 챗봇이야. 아래 [참고 자료]만 근거로 질문에 답해줘.
참고 자료에 없는 내용은 지어내지 말고 "안내 문서에서 찾을 수 없습니다"라고 답해줘.

[참고 자료]
{context}

[질문] {question}
[답변]"""

    # 3) LLM 호출
    return llm.invoke(prompt).content

# 테스트
print(rag_answer("재택근무는 일주일에 몇 번 가능해?"))

In [ ]:
# 문서에 있는 질문
print("Q1:", rag_answer("신입사원 연차는 며칠이야?"))
print()
print("Q2:", rag_answer("개발자는 노트북 메모리 몇 GB 받아?"))
print()
# 문서에 없는 질문 -> '찾을 수 없습니다' 가 나와야 정상
print("Q3:", rag_answer("회사 주차장은 몇 층까지 있어?"))

위 Q3 처럼 **문서에 없는 질문**에는 지어내지 않고 "찾을 수 없다"고 답하는지 확인하세요.
이것이 RAG가 일반 챗봇보다 **신뢰할 수 있는** 이유입니다 — 답의 근거가 *내 문서* 에 있습니다.

---
## [통합] RAG 챗봇을 Gradio 화면으로

지금까지 만든 [5]검색 + [6]생성을 **채팅 화면**에 연결합니다.
노트북 1의 챗봇과 구조는 같고, `response` 함수가 `rag_answer` 를 부르는 것만 다릅니다.

In [ ]:
import gradio as gr

def rag_chat(message, history):
    # 대화 기록(history)은 여기서는 단순화를 위해 사용하지 않고,
    # 매 질문마다 문서에서 근거를 새로 검색해 답합니다.
    return rag_answer(message, k=3)

demo = gr.ChatInterface(
    fn=rag_chat,
    type="messages",
    title="한경 테크 사내 안내 챗봇 (RAG)",
    description="신입사원 안내서를 근거로 답하는 챗봇입니다. 휴가·재택·장비·식대를 물어보세요.",
    examples=["연차는 며칠이야?", "재택근무 신청은 언제까지 해야 해?", "야근하면 저녁 식대 나와?"],
)
demo.launch()

In [ ]:
demo.close()

---
> ## ✏️ (도전 과제) 나만의 문서로 RAG 챗봇 만들기 · 직접 풀어 보세요

위 파이프라인은 **문서만 바꾸면** 어떤 주제든 챗봇이 됩니다. 직접 바꿔 보세요.

**미션**: 아래 `my_docs` 에 *여러분이 잘 아는 주제* 의 문장들을 넣고, 전체 파이프라인을 다시 만들어 질문해 보세요.
- 예: 좋아하는 게임 공략, 동아리 규칙, 우리 학과 안내, 자취 요리 레시피 등
- 각 항목은 `"제목": "본문 여러 줄"` 형태로 적으면 됩니다.

In [ ]:
# TODO 1) 나만의 문서를 채우세요 (2~5개 권장)
my_docs = {
    "주제1": """여기에 첫 번째 주제의 내용을 여러 줄로 적으세요.""",
    "주제2": """여기에 두 번째 주제의 내용을 적으세요.""",
}

# --- 아래는 위에서 배운 [1]~[4] 단계를 한 번에 실행하는 코드입니다(그대로 실행) ---
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

my_documents = [Document(page_content=text, metadata={"source": title})
                for title, text in my_docs.items()]
my_chunks = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20).split_documents(my_documents)
my_vectorstore = Chroma.from_documents(my_chunks, embedding=embeddings, collection_name="my_custom_docs")
print("나만의 벡터DB 준비 완료! 조각 수:", my_vectorstore._collection.count())

In [ ]:
# TODO 2) 나만의 문서로 답하는 함수 + 챗봇 실행
import gradio as gr

def my_rag_answer(question, k=3):
    docs = my_vectorstore.similarity_search(question, k=k)
    context = "\n\n".join(d.page_content for d in docs)
    prompt = f"""아래 [참고 자료]만 근거로 질문에 답해줘. 없는 내용은 "찾을 수 없습니다"라고 답해.

[참고 자료]
{context}

[질문] {question}
[답변]"""
    return llm.invoke(prompt).content

demo = gr.ChatInterface(
    fn=lambda message, history: my_rag_answer(message),
    type="messages",
    title="나만의 RAG 챗봇",
    description="내가 넣은 문서를 근거로 답합니다.",
)
demo.launch()

In [ ]:
demo.close()

---
## 마무리

오늘 우리는 RAG 챗봇의 **6단계 파이프라인**을 처음부터 직접 만들었습니다:
**문서 적재 → 청킹 → 임베딩 → 벡터 저장(Chroma) → 검색(코사인 top-k) → 생성**.

- 핵심 아이디어: **"내 문서에서 관련 부분을 찾아, 그걸 근거로 답한다"** → 환각이 줄고 출처가 분명해집니다.
- 3차시의 **벡터·코사인 유사도** 가 임베딩·검색에서 그대로 쓰였습니다.
- 코드 없이 같은 일을 해 주는 도구가 **Google NotebookLM** 입니다(다음 노트북에서 사용법 소개).

> 더 나아가려면(개념만): 키워드 검색을 섞는 **하이브리드 검색**, 검색 결과를 다시 정렬하는 **리랭커(re-ranker)** 등으로 품질을 높입니다. 오늘은 가장 기본인 **의미(dense) 검색** 으로 충분합니다.